import necessary libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

check the cuda device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available()  else "cpu")
device

device(type='cuda')

In [3]:
train = pd.read_csv('/kaggle/input/datasets/fedesoriano/cifar10-python-in-csv/train.csv')
test = pd.read_csv('/kaggle/input/datasets/fedesoriano/cifar10-python-in-csv/test.csv')

In [4]:
train.shape, test.shape

((50000, 3073), (10000, 3072))

In [5]:
x_train = train.iloc[:, :-1]
y_train = train.iloc[:, -1]
x_test = test.copy()

In [6]:
x_train.shape, y_train.shape, x_test.shape

((50000, 3072), (50000,), (10000, 3072))

In [7]:
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=10000, random_state = 3)

In [8]:
x_train.shape, x_val.shape, y_train.shape, y_val.shape

((40000, 3072), (10000, 3072), (40000,), (10000,))

In [9]:
class_map = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck"
}

load the pretrained model

In [10]:
from PIL import Image
class d_set(Dataset):

  def __init__(self, x: np.ndarray, y: np.ndarray, transform = None):

    self.x = x
    self.y = y
    self.transform = transform

  def __getitem__(self, idx):
    image = self.x[idx].reshape(3, 32, 32).transpose(1, 2, 0).astype(np.uint8)
    image = Image.fromarray(image)  #convert to PIL image for transformation

    if self.transform:
        image = self.transform(image)

    label = torch.tensor(self.y[idx], dtype=torch.long)

    return image, label

  def __len__(self):
    return self.x.shape[0]

In [11]:
# Function to evaluate model accuracy
def eval_model(model, loader, return_labels = False):
  model.eval()
  with torch.no_grad():
    total = 0
    correct = 0
    preds = []
    true_label = []

    for batch_x, batch_y in loader:

      batch_x, batch_y = batch_x.to(device), batch_y.to(device)
      true_label.extend(batch_y.tolist())

      y_pred = model(batch_x)
      predictions = y_pred.max(axis = 1)[1]
      preds.extend(predictions)

      total = total + batch_x.shape[0]
      correct += (predictions == batch_y).sum().item()

  accuracy = correct / total

  if return_labels:
    return accuracy, true_label, preds

  return accuracy

In [12]:
from torchvision.transforms import transforms

tf2 = transforms.Compose([
    transforms.Resize(224),  #resize to 224,224 (bilinear interpolation)
    transforms.RandomHorizontalFlip(p = 0.5), #horizontal flip
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
]
)

In [13]:
sample_size = 2000

train_set = d_set(x_train.values[:sample_size], y_train.values[:sample_size], tf2)
val_set = d_set(x_val.values[:sample_size], y_val.values[:sample_size], tf2)

#iterable dataloaders
train_loader = DataLoader(train_set, batch_size=64, shuffle = True)  #shuffle at the start of each epoch
val_loader = DataLoader(val_set, batch_size=64, shuffle = True)  #shuffle at the start of each epoch

len(train_set), len(val_set)

(2000, 2000)

The pretrained backbone already contains useful features, whereas your new classifier starts with randomly initialized weights. Therefore, you can use a higher learning rate for the classifier and a lower one for the backbone.

In [14]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

#change the output classifier layer
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([
    {
        "params": [
            p for name, p in model.named_parameters()
            if not name.startswith("fc.")
        ],
        "lr": 1e-6,
        "weight_decay": 1e-4
    },
    {
        "params": model.fc.parameters(),
        "lr": 1e-4,
        "weight_decay": 1e-4
    }
])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 185MB/s]


In [15]:
# for Early Stopping
patience = 7
patience_counter = 0
best_val_loss = float('inf')
best_model_weights = None

epochs = 200

for e in range(epochs):
  # ------------------ TRAINING PHASE ------------------
  model.train() # Keep the frozen layer in evaluation mode including its BatchNorm running statistics.
  # model.fc.train()
  train_loss = 0.0

  for batch_x, batch_y in train_loader:
      # Move data to GPU
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)

      # Forward pass
      y_pred = model(batch_x)
      loss = criterion(y_pred, batch_y)

      # Backward propagation
      optimizer.zero_grad()
      loss.backward()

      # Update weights
      optimizer.step()

      train_loss += loss.item()

  avg_train_loss = train_loss / len(train_loader)

  # ------------------ VALIDATION PHASE ------------------
  model.eval()
  val_loss = 0.0

  with torch.no_grad():
      for batch_x, batch_y in val_loader:
          batch_x, batch_y = batch_x.to(device), batch_y.to(device)
          y_pred = model(batch_x)
          loss = criterion(y_pred, batch_y)
          val_loss += loss.item()

  avg_val_loss = val_loss / len(val_loader)

  # Print epoch progress
  print(f"Epoch {e+1:02d}/{epochs:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

  # ------------------ EARLY STOPPING LOGIC ------------------
  if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      patience_counter = 0
      # Save the best model parameters
      best_model_weights = model.state_dict().copy()
      print(f"--> Val loss improved! Saving best weights.")
  else:
      patience_counter += 1
      print(f"--> No improvement in val loss ({patience_counter}/{patience})")

  if patience_counter >= patience:
      print(f"\n[Early Stopping Triggered] Stopped early at epoch {e+1}.")
      break

# Restore best weights after training stops
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
    print("Loaded best model weights into model!")

Epoch 01/200 | Train Loss: 2.3042 | Val Loss: 2.2128
--> Val loss improved! Saving best weights.
Epoch 02/200 | Train Loss: 2.1227 | Val Loss: 2.0617
--> Val loss improved! Saving best weights.
Epoch 03/200 | Train Loss: 1.9872 | Val Loss: 1.9302
--> Val loss improved! Saving best weights.
Epoch 04/200 | Train Loss: 1.8461 | Val Loss: 1.7968
--> Val loss improved! Saving best weights.
Epoch 05/200 | Train Loss: 1.7252 | Val Loss: 1.6940
--> Val loss improved! Saving best weights.
Epoch 06/200 | Train Loss: 1.6263 | Val Loss: 1.5888
--> Val loss improved! Saving best weights.
Epoch 07/200 | Train Loss: 1.5064 | Val Loss: 1.4907
--> Val loss improved! Saving best weights.
Epoch 08/200 | Train Loss: 1.4274 | Val Loss: 1.4065
--> Val loss improved! Saving best weights.
Epoch 09/200 | Train Loss: 1.3290 | Val Loss: 1.3253
--> Val loss improved! Saving best weights.
Epoch 10/200 | Train Loss: 1.2609 | Val Loss: 1.2723
--> Val loss improved! Saving best weights.
Epoch 11/200 | Train Loss: 1.1

ResNet18 backbone already has learned weights from ImageNet. When you use a learning rate of 0.001, you may be changing those useful weights too aggressively, causing the loss to start high or become unstable.

In [16]:
#evaluation on train data
train_acc = eval_model(model, train_loader)
print(f"train accuracy -> {train_acc}")

#evaluation on full valiadation data
val_set = d_set(x_val.values, y_val.values, tf2)
val_loader = DataLoader(val_set, batch_size=64, shuffle = True)
val_acc = eval_model(model, val_loader)
print(f"validation accuracy -> {val_acc}")

train accuracy -> 0.994
validation accuracy -> 0.8544


- With only 2,000 training images, the model memorized much of the training set when all pretrained parameters are allowed to update.

- However, an 86.87% validation accuracy suggests that the pretrained features are transferring well to CIFAR-10.

This is the result of training ResNet on just 2,000 samples (only 4% of the original training set)

In [ ]:
torch.save(model.state_dict(), 'resnet18_2k.pth')

## Let's train on full training data

In [18]:
sample_size = 40000

train_set = d_set(x_train.values[:sample_size], y_train.values[:sample_size], tf2)
val_set = d_set(x_val.values[:5000], y_val.values[:5000], tf2)

#iterable dataloaders
train_loader = DataLoader(train_set, batch_size=128, shuffle = True)  #shuffle at the start of each epoch
val_loader = DataLoader(val_set, batch_size=128, shuffle = True)  #shuffle at the start of each epoch

len(train_set), len(val_set)

(40000, 5000)

In [19]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)

#change the output classifier layer
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([
    {
        "params": [
            p for name, p in model.named_parameters()
            if not name.startswith("fc.")
        ],
        "lr": 1e-6,
        "weight_decay": 1e-4
    },
    {
        "params": model.fc.parameters(),
        "lr": 1e-4,
        "weight_decay": 1e-4
    }
])

In [20]:
# for Early Stopping
patience = 10
patience_counter = 0
best_val_loss = float('inf')
best_model_weights = None

epochs = 200

for e in range(epochs):
  # ------------------ TRAINING PHASE ------------------
  model.train() # Keep the frozen layer in evaluation mode including its BatchNorm running statistics.
  train_loss = 0.0

  for batch_x, batch_y in train_loader:
      # Move data to GPU
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)

      # Forward pass
      y_pred = model(batch_x)
      loss = criterion(y_pred, batch_y)

      # Backward propagation
      optimizer.zero_grad()
      loss.backward()

      # Update weights
      optimizer.step()

      train_loss += loss.item()

  avg_train_loss = train_loss / len(train_loader)

  # ------------------ VALIDATION PHASE ------------------
  model.eval()
  val_loss = 0.0

  with torch.no_grad():
      for batch_x, batch_y in val_loader:
          batch_x, batch_y = batch_x.to(device), batch_y.to(device)
          y_pred = model(batch_x)
          loss = criterion(y_pred, batch_y)
          val_loss += loss.item()

  avg_val_loss = val_loss / len(val_loader)

  # Print epoch progress
  print(f"Epoch {e+1:02d}/{epochs:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

  # ------------------ EARLY STOPPING LOGIC ------------------
  if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      patience_counter = 0
      # Save the best model parameters
      best_model_weights = model.state_dict().copy()
      print(f"--> Val loss improved! Saving best weights.")
  else:
      patience_counter += 1
      print(f"--> No improvement in val loss ({patience_counter}/{patience})")

  if patience_counter >= patience:
      print(f"\n[Early Stopping Triggered] Stopped early at epoch {e+1}.")
      break

# Restore best weights after training stops
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
    print("Loaded best model weights into model!")

Epoch 01/200 | Train Loss: 1.6347 | Val Loss: 1.1025
--> Val loss improved! Saving best weights.
Epoch 02/200 | Train Loss: 0.8845 | Val Loss: 0.7160
--> Val loss improved! Saving best weights.
Epoch 03/200 | Train Loss: 0.6285 | Val Loss: 0.5543
--> Val loss improved! Saving best weights.
Epoch 04/200 | Train Loss: 0.5082 | Val Loss: 0.4618
--> Val loss improved! Saving best weights.
Epoch 05/200 | Train Loss: 0.4406 | Val Loss: 0.4080
--> Val loss improved! Saving best weights.
Epoch 06/200 | Train Loss: 0.3894 | Val Loss: 0.3712
--> Val loss improved! Saving best weights.
Epoch 07/200 | Train Loss: 0.3547 | Val Loss: 0.3405
--> Val loss improved! Saving best weights.
Epoch 08/200 | Train Loss: 0.3251 | Val Loss: 0.3308
--> Val loss improved! Saving best weights.
Epoch 09/200 | Train Loss: 0.3074 | Val Loss: 0.3088
--> Val loss improved! Saving best weights.
Epoch 10/200 | Train Loss: 0.2868 | Val Loss: 0.2878
--> Val loss improved! Saving best weights.
Epoch 11/200 | Train Loss: 0.2

In [21]:
#evaluation on train data
train_acc = eval_model(model, train_loader)
print(f"train accuracy -> {train_acc}")

#evaluation on full valiadation data
val_set = d_set(x_val.values, y_val.values, tf2)
val_loader = DataLoader(val_set, batch_size=64, shuffle = True)
val_acc = eval_model(model, val_loader)
print(f"validation accuracy -> {val_acc}")

train accuracy -> 0.995975
validation accuracy -> 0.9381


In [ ]:
torch.save(model.state_dict(), 'resnet18_40k.pth')

---------------------------------------------------------------- **over** ---------------------------------------------------------